# Forecast-period atmospheric ensemble model evaluation

Reads selected ensemble fields and references only when their diagnostic cache is missing or forced, computes extended verification metrics, then plots selected cached maps.


In [ ]:
from pathlib import Path
INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_out")

workflow_name = "fcst_atm_model_evaluation"
component = "atm"
run_segment = "fc"
season = "S0"
experiments = ("CTRL10-S0", "CAPT10-S0", "DART20-S0", "DART40-S0")
variables = ("TS", "TREFHT", "PSL")
plot_metrics = ("bias", "rmse", "spread", "crps")
lat_range = (-90, 90)
lon_range = (-180, 180)
max_members = None
n_bootstrap = 0
include_rank_histogram = True
random_seed = 42
force_compute = False
show = True


In [ ]:
import sys
import pandas as pd
import xarray as xr
repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.output_paths import workflow_output_dirs
from util.bias_metrics import ATM_VARIABLES, LND_VARIABLES, experiment_registry, read_model_ensemble, read_reference
from util.model_evaluation import MAP_METRICS, SCALAR_METRICS, OPTIONAL_METRICS, evaluation_cache_path, cache_evaluation, plot_evaluation_maps


## Notes: available variables and metrics

Atmosphere variables: `FLUT` (CERES-OAFlux), `PRECT` (GPCP), `TMQ`, `TS`, `TREFHT`, and `PSL` (ERA5). Land variables: `TSA` (MODIS LST), `H2OSOI` top 5 cm (ESA CCI), and `SOILWATER_10CM` (CPC soil moisture).

Map metrics: `mean`, `reference_mean`, `bias`, `bias_stddev`, `rmse`, `spread`, `crps`, and `coverage`. Scalar metrics: `model_global_mean`, `reference_global_mean`, `model_rms`, `reference_rms`, `rmse_global`, `pattern_correlation`, `bias_global`, `spread_global`, `crps_global`, `coverage_global`, `ssr`, `bias_squared`, `spread_squared`, `mse`, and `sem`. Optional: `rank_histogram`. With `n_bootstrap > 0`, bias/RMSE/spread/CRPS confidence bounds are cached too.

Existing request-keyed NetCDF files are reused. Set `force_compute = True` to reread raw model/reference data and rebuild them.


In [ ]:
component_variables = ATM_VARIABLES if component == "atm" else LND_VARIABLES
data_dir, figure_dir = workflow_output_dirs(workflow_name, output_root=DIAGNOSTIC_OUTPUT_ROOT)
registry = experiment_registry(INPUT_DATA_ROOT, season, run_segment)
unknown = set(experiments) - set(registry)
if unknown:
    raise ValueError(f"Unavailable experiments: {sorted(unknown)}; choose from {list(registry)}")
all_results = {}
for variable in variables:
    if variable not in component_variables:
        raise ValueError(f"{variable} unavailable for {component}; choose from {list(component_variables)}")
    variable_results = {}
    for experiment in experiments:
        period = registry[experiment]["period"]
        target = evaluation_cache_path(data_dir, experiment, variable, period)
        if target.is_file() and not force_compute:
            ds = cache_evaluation(target, n_bootstrap=n_bootstrap)
            print(f"[REUSE] {target}")
        else:
            model, meta = read_model_ensemble(INPUT_DATA_ROOT, experiment, season, run_segment, component, variable, period=period, lat_range=lat_range, lon_range=lon_range, max_members=max_members)
            reference = read_reference(INPUT_DATA_ROOT / "reference", component, variable, period, lat_range=lat_range, lon_range=lon_range)
            ds = cache_evaluation(target, model=model, reference=reference, force_compute=force_compute, experiment=experiment, variable=variable, component=component, run_segment=run_segment, season=season, period=period, n_bootstrap=n_bootstrap, random_seed=random_seed, include_rank_histogram=include_rank_histogram)
            print(f"[WRITE] {target}")
        variable_results[experiment] = ds
    all_results[variable] = variable_results
    for metric in plot_metrics:
        fig = plot_evaluation_maps(variable_results, metric, variable, label_map={name: registry[name]["label"] for name in experiments}, title=f"{variable}: {metric} ({run_segment})")
        output = figure_dir / f"{variable}_{metric}_{season}_{run_segment}.png"
        fig.savefig(output, dpi=200, bbox_inches="tight")
        if not show: import matplotlib.pyplot as plt; plt.close(fig)
        print(f"[FIGURE] {output}")
rows=[]
for variable, results in all_results.items():
    for experiment, ds in results.items():
        row={"variable":variable,"experiment":experiment}
        for metric in SCALAR_METRICS:
            if metric in ds: row[metric]=float(ds[metric])
        rows.append(row)
summary=pd.DataFrame(rows)
summary
